In [6]:
import tensorflow as tf
import numpy as np
import cv2
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG19
from tensorflow.keras.callbacks import Callback, LearningRateScheduler, EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

# Define dataset path & parameters
data_dir = "E:/Projects/Strabismus/Denoised"
img_height, img_width = 224, 224
batch_size = 64
num_classes = 5  

# Function to apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
def apply_clahe(img):
    img_lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(img_lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_lab = cv2.merge((l, a, b))
    return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)

# Data augmentation & preprocessing
datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.vgg19.preprocess_input,
    validation_split=0.15,
    rotation_range=40,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    zoom_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

def preprocess_input(img):
    img = apply_clahe(img)
    return img

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=(img_height, img_width),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

# Compute class weights
class_weights = compute_class_weight('balanced', classes=np.arange(num_classes), y=train_generator.classes)
class_weight_dict = dict(enumerate(class_weights))

# Load & modify VGG19 model
base_model = VGG19(weights='imagenet', include_top=False, input_shape=(img_height, img_width, 3))
base_model.trainable = False  

# Define custom model
model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dense(512, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    Dropout(0.5),
    Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    Dropout(0.5),
    Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

# Define learning rate scheduler
def lr_scheduler(epoch, lr):
    if epoch < 5:
        return lr
    elif epoch < 15:
        return lr * 0.5
    else:
        return lr * 0.1

callback_lr = LearningRateScheduler(lr_scheduler)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)

# Use AdamW Optimizer (Instead of Lookahead + RAdam)
optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=0.001)

# Compile model with better loss function
model.compile(optimizer=optimizer, 
              loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1), 
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Precision(name='precision')])

# Define custom metrics callback
class MetricsCallback(Callback):
    def on_epoch_end(self, epoch, logs=None):
        y_pred_probs = model.predict(val_generator, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        y_true = val_generator.classes[:len(y_pred)]
        
        acc = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, average='macro', zero_division=1)
        recall = recall_score(y_true, y_pred, average='macro', zero_division=1)
        f1 = f1_score(y_true, y_pred, average='macro', zero_division=1)
        
        print(f'\nEpoch {epoch+1} Metrics: Accuracy: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1 Score: {f1:.4f}')

# Define early stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model (First Phase - Frozen Base Model)
model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[MetricsCallback(), callback_lr, reduce_lr, early_stopping]
)

# Progressive Layer Unfreezing Strategy
for layer in base_model.layers[:10]:  
    layer.trainable = False
for layer in base_model.layers[10:]:  
    layer.trainable = True

# Recompile model with a lower learning rate for fine-tuning
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), 
              loss='categorical_crossentropy', 
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])

# Continue training with fine-tuning
model.fit(
    train_generator,
    epochs=30,
    validation_data=val_generator,
    class_weight=class_weight_dict,
    callbacks=[MetricsCallback(), callback_lr, reduce_lr, early_stopping]
)

# Implement Test-Time Augmentation (TTA) for inference
def tta_predict(model, image):
    aug_images = [datagen.random_transform(image) for _ in range(5)]
    preds = np.mean([model.predict(img[np.newaxis, :]) for img in aug_images], axis=0)
    return preds

print("Model training and optimization complete.")


Found 4395 images belonging to 5 classes.
Found 775 images belonging to 5 classes.
Epoch 1/20
69/69 [==============================] - ETA: 0s - loss: 2.9752 - accuracy: 0.2617 - auc: 0.5589 - precision: 0.2900
Epoch 1 Metrics: Accuracy: 0.2219, Precision: 0.2078, Recall: 0.2197, F1 Score: 0.1974
69/69 [==============================] - 46s 522ms/step - loss: 2.9752 - accuracy: 0.2617 - auc: 0.5589 - precision: 0.2900 - val_loss: 2.3900 - val_accuracy: 0.3884 - val_auc: 0.7094 - val_precision: 0.9600 - lr: 0.0010
Epoch 2/20
69/69 [==============================] - ETA: 0s - loss: 2.3953 - accuracy: 0.3311 - auc: 0.6447 - precision: 0.6370
Epoch 2 Metrics: Accuracy: 0.1858, Precision: 0.1571, Recall: 0.1860, F1 Score: 0.1321
69/69 [==============================] - 38s 556ms/step - loss: 2.3953 - accuracy: 0.3311 - auc: 0.6447 - precision: 0.6370 - val_loss: 2.2599 - val_accuracy: 0.4052 - val_auc: 0.7268 - val_precision: 0.9672 - lr: 0.0010
Epoch 3/20
69/69 [===========================